# Phase 2c-v2 — Causal patch SWEEP
Phase 2c: patching up-block self-attention at mid steps did NOT move the count (readout != control). Here we sweep the patch over **attention type (attn1 image-side vs attn2 matching)**, **timing (early 0-5 vs mid)**, and **block (up vs mid)** to find where injecting the donor actually moves the output count. Whichever config makes the count follow the donor = the causal control site.

**Runtime:** GPU (~25-35 min).

In [ ]:
import os
if not os.path.exists('src'):
    !git clone https://github.com/serinaqin/T2I-Count-Anomaly.git
    %cd T2I-Count-Anomaly
!pip install -q -r requirements.txt
!pip install -q pytest groundingdino-py

In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np, pandas as pd, os, yaml
from src.prompts import build_prompt
from src.pipeline import (load_sdxl, generate, catalog_attention_sites,
                          generate_and_capture, raw_reducer, generate_with_patch)
from src.detector import Detector
from src.scoring import count_from_detections
from src.config import load_config

In [ ]:
cfg = load_config('configs/phase2c2.yaml')
raw = yaml.safe_load(open('configs/phase2c2.yaml'))
pairs, sweep = raw['pairs'], raw['sweep']
cap_blocks, cap_steps = raw['capture_blocks'], raw['capture_steps']
obj = cfg.objects[0]
for s in sweep: print(s)

In [ ]:
pipe = load_sdxl()
det = Detector()
# capture attn1 AND attn2 (first transformer block) in the capture blocks
cap_sites = [s for s in catalog_attention_sites(pipe.unet)
             if 'transformer_blocks.0.' in s
             and (s.endswith('attn1') or s.endswith('attn2'))
             and any(b in s for b in cap_blocks)]
print(len(cap_sites), 'capture sites:'); print(cap_sites)
def cnt(img):
    return count_from_detections(det.detect(img, [obj]), obj, cfg.score_threshold)
def sites_for(entry):
    suf = ('attn1', 'attn2') if entry['attn'] == 'both' else (entry['attn'],)
    return [s for s in cap_sites if entry['block'] in s and s.endswith(suf)]

In [ ]:
# Capture donor once per (pair, seed) over the superset; patch each subset.
rows = []
for src, dnr in pairs:
    sp, dp = build_prompt(src, obj), build_prompt(dnr, obj)
    direction = 'up' if src < dnr else 'down'
    for seed in cfg.seeds:
        _, snaps = generate_and_capture(pipe, dp, seed, cap_sites, cap_steps,
                                        cfg.num_inference_steps, reducer=raw_reducer)
        c_base = cnt(generate(pipe, sp, seed, cfg.num_inference_steps))
        c_donor = cnt(generate(pipe, dp, seed, cfg.num_inference_steps))
        for e in sweep:
            es, esite = set(e['steps']), set(sites_for(e))
            pm = {st: {s: snaps[st][s] for s in snaps[st] if s in esite}
                  for st in snaps if st in es}
            c_patch = cnt(generate_with_patch(pipe, sp, seed, pm, cfg.num_inference_steps))
            rows.append({'config': e['name'], 'direction': direction, 'seed': seed,
                         'c_donor': c_donor, 'c_base': c_base, 'c_patch': c_patch,
                         'delta': c_patch - c_base})
        print(f'{direction} seed {seed} done')
df = pd.DataFrame(rows)
os.makedirs('results', exist_ok=True)
df.to_csv('results/phase2c2_sweep.csv', index=False)
df.head(12)

In [ ]:
# Donor-directed effect: up expects delta>0, down expects delta<0.
def signed(g, direction):
    return g['delta'] if direction == 'up' else -g['delta']
summ = []
for (config, direction), g in df.groupby(['config', 'direction']):
    s = signed(g, direction)
    summ.append({'config': config, 'direction': direction,
                 'donor_directed_delta': s.mean(),
                 'frac_expected': (s > 0).mean(),
                 'base': g.c_base.mean(), 'patch': g.c_patch.mean(),
                 'donor': g.c_donor.mean()})
summ = pd.DataFrame(summ).sort_values('donor_directed_delta', ascending=False)
summ

In [ ]:
# Bar: mean donor-directed delta per config, split by direction.
import matplotlib.pyplot as plt
piv = summ.pivot(index='config', columns='direction', values='donor_directed_delta')
ax = piv.plot(kind='bar', figsize=(9, 5))
ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel('donor-directed delta  (positive = moved toward donor)')
ax.set_title('Causal patch sweep: which site/timing controls the count?')
plt.tight_layout()
plt.savefig('results/phase2c2_sweep.png', dpi=100, bbox_inches='tight'); plt.show()

## How to read this
`donor_directed_delta` = how far the patched count moved TOWARD the donor (positive good, in BOTH directions). `frac_expected` = fraction of seeds that moved the right way.

- **A config with clearly positive donor-directed delta in BOTH up and down** = the causal control site/timing. That (block, attn-type, steps) is where the count is set -> Phase 4 mitigation target.
- **attn2 configs win** = the text->image MATCHING (cross-attention) sets the count (your leading hypothesis).
- **early-step configs win** = the count is committed at high noise, early; mid-step patching was simply too late.
- **Nothing moves it (all ~0)** = the count is not controlled anywhere in these blocks/steps -> it is locked by the INITIAL NOISE; next test is a noise-swap (change x_T, keep prompt) to confirm the count follows the seed.